# 04 — Consumer artifact bundle

Thin apps read **pre-emitted JSON bundles** — no browser-side policy.

```text
cm compile / cm impact → bundle.manifest.json → catalog-viewer / lineage-explorer
```

| Bundle | Viewer | Impact focus |
|--------|--------|-------------|
| `minimal` | catalog-viewer | upstream from `orders_base.amount` |
| `lineage-demo` | lineage-explorer | downstream from `orders_base.amount` |

Viewers run locally via `python -m http.server` — see [consumers README](../consumers/README.md). In Colab we inspect bundle JSON in-notebook.

In [ ]:
import importlib.util
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from types import ModuleType

_COLD_START_GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/ClearMetric-Labs/ClearMetric-Core/main"
)

_CACHED_NOTEBOOKS_DIR = Path(
    "/Users/jonkim/.cache/clearmetric/github-main/examples/notebooks"
)


def _github_raw_base() -> str:
    paths = sys.modules.get("_paths")
    default = (
        paths.GITHUB_RAW_BASE if paths is not None else _COLD_START_GITHUB_RAW_BASE
    )
    return os.environ.get("CM_CLEARMETRIC_GITHUB_RAW_BASE", default)


def _fetch_repo_file(repo_relative: str, dest: Path) -> None:
    if dest.is_file():
        return
    paths = sys.modules.get("_paths")
    if paths is not None:
        paths._fetch_github_file(repo_relative, dest)
        return
    url = f"{_github_raw_base()}/{repo_relative}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            dest.write_bytes(response.read())
    except urllib.error.URLError as exc:
        raise FileNotFoundError(
            f"Failed to download {url}. Check network access and branch path."
        ) from exc


def _find_local_helpers(start: Path | None = None) -> Path | None:
    start = start or Path.cwd()
    for root in (start, *start.parents):
        nested = root / "examples" / "notebooks"
        if (nested / "_paths.py").is_file():
            return nested
        if (root / "_paths.py").is_file() and (root / "_notebook_setup.py").is_file():
            return root
    return None


def _resolve_setup_file(start: Path | None = None) -> Path:
    local = _find_local_helpers(start)
    if local is not None:
        return local / "_notebook_setup.py"
    paths = sys.modules.get("_paths")
    cache_dir = (
        paths.CACHED_NOTEBOOKS_DIR if paths is not None else _CACHED_NOTEBOOKS_DIR
    )
    setup_file = cache_dir / "_notebook_setup.py"
    if not setup_file.is_file():
        _fetch_repo_file("examples/notebooks/_notebook_setup.py", setup_file)
    return setup_file


def _exec_setup_module(setup_file: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location("_notebook_setup", setup_file)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {setup_file}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


_exec_setup_module(_resolve_setup_file()).setup_notebook()

In [ ]:
import json

from _paths import (
    build_bundle_script,
    consumer_bundle_dir,
    consumer_scenario,
    lineage_demo_project,
    load_checks_runner,
    repo_root,
)

BUNDLE_DIR = consumer_bundle_dir()
LINEAGE_BUNDLE_DIR = consumer_bundle_dir("lineage-demo")
SCENARIO = consumer_scenario()
SCENARIO_PROJECT = lineage_demo_project()
print(f"minimal bundle: {BUNDLE_DIR}")
print(f"scenario project: {SCENARIO_PROJECT}")

## Manifest tour (`minimal`)

In [ ]:
from clearmetric.core.validate import (
    load_artifact_file,
    load_bundle_manifest_file,
    load_impact_output_file,
)

manifest = load_bundle_manifest_file(BUNDLE_DIR / "bundle.manifest.json")
print(
    json.dumps(
        {
            "scenario_id": manifest["scenario_id"],
            "lane": manifest["artifacts"]["graph"]["lane"],
            "artifacts": list(manifest["artifacts"].keys()),
            "default_impact": manifest["defaults"]["impact_key"],
        },
        indent=2,
    )
)

graph = load_artifact_file(BUNDLE_DIR / manifest["artifacts"]["graph"]["path"])
catalog = load_artifact_file(BUNDLE_DIR / manifest["artifacts"]["catalog"]["path"])
print(f"graph nodes: {len(graph.nodes)}  catalog nodes: {len(catalog.nodes)}")

## Default impact artifact

In [ ]:
impact_key = manifest["defaults"]["impact_key"]
impact_ref = manifest["artifacts"]["impacts"][impact_key]
impact = load_impact_output_file(BUNDLE_DIR / impact_ref["path"])
print(f"selection_id: {impact['selection_id']}")
print(f"related_ids: {impact['related_ids']}")
print(f"warnings: {[w['code'] for w in impact['warnings']]}")

## `lineage-demo` bundle (downstream trace)

In [ ]:
lineage_manifest = load_bundle_manifest_file(
    LINEAGE_BUNDLE_DIR / "bundle.manifest.json"
)
lineage_key = lineage_manifest["defaults"]["impact_key"]
lineage_ref = lineage_manifest["artifacts"]["impacts"][lineage_key]
lineage_impact = load_impact_output_file(LINEAGE_BUNDLE_DIR / lineage_ref["path"])
print(f"impact: {lineage_key}")
for node_id in lineage_impact["related_ids"]:
    print(f"  {node_id}")

## Corpus checks (centralized runner)

In [ ]:
checks = load_checks_runner()
violations = checks.run_checks(BUNDLE_DIR, SCENARIO / "checks.yaml")
if violations:
    raise AssertionError("\n".join(violations))
print("minimal corpus checks passed")

## Optional: regenerate bundle (full clone only)

Requires a local checkout with `scripts/consumers/build_bundle.py`.

In [ ]:
import subprocess

try:
    script = build_bundle_script()
except FileNotFoundError as exc:
    print(f"skip regenerate: {exc}")
else:
    import tempfile

    with tempfile.TemporaryDirectory() as tmp:
        out = Path(tmp) / "bundle"
        result = subprocess.run(
            [
                sys.executable,
                str(script),
                "--scenario",
                str(SCENARIO),
                "--out",
                str(out),
            ],
            capture_output=True,
            text=True,
            cwd=str(repo_root()),
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr or result.stdout)
        print(result.stdout.strip())
        print("rebuilt bundle validates OK")